In [ ]:

import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL    = "https://github.com/Gianbattistabsn/FAIML-RL-26.git"
    REPO_BRANCH = "alessandro-PPO-SAC"
    REPO_ROOT   = "/content/FAIML-RL-26"

    if not os.path.exists(REPO_ROOT):
        result = subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_ROOT],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print("Branch clone failed, falling back to clone + checkout")
            print("git stderr:", result.stderr.strip())
            subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)
            subprocess.run(["git", "-C", REPO_ROOT, "checkout", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "-C", REPO_ROOT, "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", REPO_ROOT, "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "origin", REPO_BRANCH], check=True)

    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], capture_output=True)

else:
    # In VS Code notebooks, __vsc_ipynb_file__ holds the absolute path of the .ipynb file.
    # clone_colab.ipynb lives in part2/ -> two dirname() calls up is REPO_ROOT.
    try:
        REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__vsc_ipynb_file__)))
    except NameError:
        # Fallback: walk upward from cwd until we find a folder containing part2/rand_wrapper.py
        _candidate = os.path.abspath(os.getcwd())
        for _ in range(8):
            if os.path.isfile(os.path.join(_candidate, "part2", "rand_wrapper.py")):
                REPO_ROOT = _candidate
                break
            _candidate = os.path.dirname(_candidate)
        else:
            raise RuntimeError(
                f"Could not locate repo root from cwd={os.getcwd()}.\n"
                "Make sure you run from inside the FAIML-RL-26 repo."
            )

os.chdir(REPO_ROOT)

# part2/ contains rand_wrapper.py and other local modules.
# part2/panda-gym/ is the local panda_gym build that supports the 'type' kwarg —
# it must come BEFORE anything on sys.path so it shadows the pip-installed version.
PART2_DIR       = os.path.join(REPO_ROOT, "part2")
LOCAL_PANDA_GYM = os.path.join(PART2_DIR, "panda-gym")
for p in (LOCAL_PANDA_GYM, PART2_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

# 1) Base requirements (mujoco may fail on some platforms – that's OK, we use pybullet)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=False)

# 2) pybullet (required by panda-gym; not listed in requirements.txt)
subprocess.run([sys.executable, "-m", "pip", "install", "pybullet", "-q"], check=False)

# 3) Editable install of the local panda-gym so its environments get registered
subprocess.run([sys.executable, "-m", "pip", "install", "-e", LOCAL_PANDA_GYM, "-q"], check=False)

# 4) On Colab: replace opencv-python with headless build AFTER requirements.txt
#    to avoid the AttributeError in stable_baselines3's atari_wrappers at import time.
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "opencv-python-headless", "-q"],
        check=False,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "pyvirtualdisplay", "-q"], check=False)

print(f"Mode      : {'Colab' if IN_COLAB else 'Local'}")
print(f"Repo root : {REPO_ROOT}")
print(f"part2 dir : {PART2_DIR}  (exists: {os.path.isdir(PART2_DIR)})")
print(f"rand_wrapper found : {os.path.isfile(os.path.join(PART2_DIR, 'rand_wrapper.py'))}")

if IN_COLAB:
    branch = subprocess.run(
        ["git", "-C", REPO_ROOT, "branch", "--show-current"],
        capture_output=True, text=True
    ).stdout.strip()
    print(f"Branch    : {branch}")
    if branch != REPO_BRANCH:
        raise RuntimeError(f"Wrong branch! Expected '{REPO_BRANCH}', got '{branch}'")


In [ ]:

# --- Sanity check: verify all key imports work ---
import gymnasium as gym
import numpy as np
import torch
import panda_gym  # registers PandaPush-v3 etc.
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback
from rand_wrapper import RandomizationWrapper

print("All imports OK")
print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
print(f"Registered panda envs: {[e for e in gym.envs.registry if 'Panda' in e][:5]}")


## Training Configuration

Edit the variables below, then run the Training cell.


In [ ]:

# ── Adjust these before running ──────────────────────────────────────────────
SAMPLING_STRATEGY = "none"    # "none" | "udr" | "adr"
ENV_TYPE          = "source"  # "source" | "target"
TIMESTEPS         = 200_000   # total training steps
LOAD_MODEL        = False     # True → load existing zip instead of training
# ─────────────────────────────────────────────────────────────────────────────


## Train SAC on PandaPush-v3


In [ ]:

import os

save_name = os.path.join(
    REPO_ROOT, "part2", "models",
    f"sac_push_{SAMPLING_STRATEGY}_{ENV_TYPE}_{TIMESTEPS // 1000}k"
)
os.makedirs(os.path.dirname(save_name), exist_ok=True)

env = gym.make("PandaPush-v3", render_mode="rgb_array", type=ENV_TYPE, reward_type="dense")
if SAMPLING_STRATEGY != "none":
    env = RandomizationWrapper(env, mode=SAMPLING_STRATEGY)

if LOAD_MODEL:
    model = SAC.load(f"{save_name}.zip")
    model.set_env(DummyVecEnv([lambda: env]))
    print(f"Model loaded from {save_name}.zip")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Training on device: {device}")

    vec_env = DummyVecEnv([lambda: env])

    model = SAC(
        policy="MultiInputPolicy",
        env=vec_env,
        device=device,
        verbose=1,
        learning_rate=1e-3,
        buffer_size=200_000,
        batch_size=256,
        tensorboard_log=f"{save_name}/logs",
    )

    checkpoint_cb = CheckpointCallback(
        save_freq=50_000,
        save_path=f"{save_name}/checkpoints",
        name_prefix="model",
    )

    model.learn(total_timesteps=TIMESTEPS, callback=checkpoint_cb, progress_bar=True)
    model.save(save_name)
    print(f"Model saved to {save_name}.zip")


## Evaluate the Trained Model


In [ ]:

N_EVAL_EPISODES = 20

eval_env = gym.make("PandaPush-v3", render_mode="rgb_array", type=ENV_TYPE, reward_type="dense")

episode_returns = []
successes = []

for ep in range(1, N_EVAL_EPISODES + 1):
    obs, info = eval_env.reset()
    done = False
    ep_return = 0.0

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        ep_return += float(reward)
        done = terminated or truncated

    episode_returns.append(ep_return)
    if isinstance(info, dict) and "is_success" in info:
        successes.append(float(info["is_success"]))
    print(f"Episode {ep:03d} | return = {ep_return:.3f}")

eval_env.close()

returns = np.array(episode_returns)
print(f"\n=== Eval over {N_EVAL_EPISODES} episodes ===")
print(f"Mean return : {returns.mean():.3f} ± {returns.std():.3f}")
print(f"Min / Max   : {returns.min():.3f} / {returns.max():.3f}")
if successes:
    print(f"Success rate: {np.mean(successes):.2%}")
